In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine, text


# database connections 


In [ ]:
with open('../../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['OLTP']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)


# Extract


In [ ]:
dim_tipo_novedad = pd.read_sql_table('mensajeria_tiponovedad', co_sa)
dim_tipo_novedad


# Transformations


In [ ]:
dim_tipo_novedad.rename(columns={'id': 'id_tipo_novedad'}, inplace=True)
dim_tipo_novedad.replace({np.nan: 'NO APLICA', ' ': 'NO APLICA', '': 'NO APLICA'}, inplace=True)
unknown = pd.DataFrame([{'id_tipo_novedad': -1, 'nombre': 'NO APLICA'}])
dim_tipo_novedad = pd.concat([unknown, dim_tipo_novedad], ignore_index=True)
dim_tipo_novedad["saved"] = date.today()
dim_tipo_novedad


# load


In [ ]:
with etl_conn.begin() as conn:
    conn.execute(text('Delete from dim_tipo_novedad'))
dim_tipo_novedad.to_sql('dim_tipo_novedad', etl_conn, if_exists='append', index=False)
